In [ ]:
import pandas as pd
import numpy as np
import re

# Load the dataset first
df = pd.read_csv('/content/Rent_Board_Housing_Inventory_20261006.csv')

# Keep the original dataset unchanged
Uncleaned = df.copy()

### Standardize missing values

In [ ]:
cleaned = cleaned.replace(
    ["Unknown", "unknown", "", "NULL", "null", "N/A", "NA"],
    np.nan
)

### Convert date columns

In [ ]:
# Convert date columns (only if they exist in the DataFrame)
date_columns = [
    "occupancy_or_vacancy_date",
    "vacancy_date",
    "signature_date",
    "data_as_of",
    "data_loaded_at"
]

for col in date_columns:
    if col in cleaned.columns:
        cleaned[col] = pd.to_datetime(
            cleaned[col],
            errors="coerce"
        )

### Drop columns not needed for the main analysis

In [ ]:
columns_to_drop = [
    "case_type_name",
    "data_as_of",
    "data_loaded_at",
    "occupancy_or_vacancy_date_history",
    "base_rent_includes_other_utilities"
]

# Initialize 'cleaned' from the dataset
cleaned = df.copy()

cleaned = cleaned.drop(
    columns=columns_to_drop,
    errors="ignore"
)

### Clean property year and create age

In [ ]:
# Invalid years become missing
cleaned.loc[
    (cleaned["year_property_built"] < 1800) |
    (cleaned["year_property_built"] > cleaned["submission_year"]),
    "year_property_built"
] = np.nan

cleaned["property_age"] = (
    cleaned["submission_year"] -
    cleaned["year_property_built"]
)

### Remove non-residential records

In [ ]:
# Remove non-residential records
cleaned = cleaned[
    cleaned["occupancy_type"].isin([
        "Occupied by non-owner",
        "Occupied by owner",
        "Vacant"
    ])
].copy()


### Keep block numbers as text because leading zeros matter

In [ ]:
# Keep block numbers as text because leading zeros matter
cleaned["block_num"] = cleaned["block_num"].astype("string")

### Convert numeric columns

In [ ]:
# Convert numeric columns
for col in [
    "unit_count",
    "submission_year",
    "year_property_built"
]:
    cleaned[col] = pd.to_numeric(
        cleaned[col],
        errors="coerce"
    )

### Treat supervisor district as a category

In [ ]:
# Treat supervisor district as a category
cleaned["supervisor_district"] = (
    cleaned["supervisor_district"].astype("string")
)

### Convert utility columns

In [ ]:
utility_columns = [
    "base_rent_includes_water_sewer",
    "base_rent_includes_natural_gas",
    "base_rent_includes_electricity",
    "base_rent_includes_refuse_recycling",
    "base_rent_includes_other_utilities"
]

for col in utility_columns:
    if col in cleaned.columns:
        cleaned[col] = cleaned[col].map({
            "Y": 1,
            "N": 0
        })

### Clean Bedroom Categories

In [ ]:
# Preserve the original values
cleaned["bedroom_count_original"] = cleaned["bedroom_count"]

def clean_bedroom_category(value):
    if pd.isna(value):
        return "Missing"

    value = str(value).lower().strip()

    if "studio" in value or value == "0":
        return "Studio"
    elif value in ["1", "one-bedroom", "one bedroom"]:
        return "1 Bedroom"
    elif value in ["2", "two-bedroom", "two bedroom"]:
        return "2 Bedrooms"
    elif value in ["3", "three-bedroom", "three bedroom"]:
        return "3 Bedrooms"
    elif value in ["4", "four-bedroom", "four bedroom"]:
        return "4 Bedrooms"
    elif value in ["5+"]:
        return "5+ Bedrooms"
    else:
        return "Other"

cleaned["bedroom_category"] = (
    cleaned["bedroom_count"]
    .apply(clean_bedroom_category)
)

print(cleaned["bedroom_category"].value_counts())

bedroom_category
1 Bedroom      210986
Studio         142198
2 Bedrooms     128847
3 Bedrooms      30355
Missing         28393
4 Bedrooms       6255
5+ Bedrooms      1612
Other             273
Name: count, dtype: int64


### Clean Bathroom Categories

In [ ]:
# Preserve the original values
cleaned["bathroom_count_original"] = cleaned["bathroom_count"]

def clean_bathroom_category(value):
    if pd.isna(value):
        return "Missing"

    value = str(value).lower().strip()

    if "shared" in value:
        return "Shared"
    elif "one and a half" in value:
        return "1.5 Bathrooms"
    elif "two and a half" in value:
        return "2.5 Bathrooms"
    elif "three" in value or value == "3":
        return "3+ Bathrooms"
    elif "two" in value or value == "2":
        return "2 Bathrooms"
    elif "one" in value or value == "1":
        return "1 Bathroom"
    else:
        return "Other"

cleaned["bathroom_category"] = (
    cleaned["bathroom_count"]
    .apply(clean_bathroom_category)
)

print(cleaned["bathroom_category"].value_counts())

bathroom_category
1 Bathroom       418998
2 Bathrooms       60853
Missing           28196
Shared            21020
1.5 Bathrooms     10902
3+ Bathrooms       4689
2.5 Bathrooms      3912
Other               349
Name: count, dtype: int64


### Deriving street name from address

In [ ]:
# Derive street name from values such as:
# "400 Block of STOCKTON ST"

cleaned["street_name"] = (
    cleaned["block_address"]
    .astype("string")
    .str.replace(
        r"^\s*\d+\s+Block\s+of\s+",
        "",
        regex=True,
        case=False
    )
    .str.strip()
    .str.upper()
)

# Check the result
display(
    cleaned[["block_address", "street_name"]].head(10)
)

# Check missing street names
print("Missing street names:", cleaned["street_name"].isna().sum())

# Drop the original masked address
cleaned = cleaned.drop(columns=["block_address"])

,block_address,street_name
0,300 Block of BRIGHTON AVE,BRIGHTON AVE
1,300 Block of BRIGHTON AVE,BRIGHTON AVE
2,0 Block of ROMOLO ST,ROMOLO ST
3,400 Block of STOCKTON ST,STOCKTON ST
4,300 Block of BRIGHTON AVE,BRIGHTON AVE
5,300 Block of BRIGHTON AVE,BRIGHTON AVE
6,300 Block of BRIGHTON AVE,BRIGHTON AVE
7,200 Block of N LAKE MERCED HLS,N LAKE MERCED HLS
8,500 Block of CRESCENT CT,CRESCENT CT
9,400 Block of STOCKTON ST,STOCKTON ST


Missing street names: 76


### Extract Latitude and Longitude for MAP

In [ ]:
import json
import ast
import re
import numpy as np

def extract_coordinates(value):
    if pd.isna(value):
        return pd.Series([np.nan, np.nan])

    text = str(value)

    # Format such as:
    # {"type": "Point", "coordinates": [-122.4, 37.7]}
    try:
        point_data = json.loads(text.replace("'", '"'))
        longitude, latitude = point_data["coordinates"]
        return pd.Series([longitude, latitude])
    except:
        pass

    # Format such as:
    # POINT (-122.4 37.7)
    match = re.search(
        r"POINT\s*\(\s*(-?\d+\.\d+)\s+(-?\d+\.\d+)\s*\)",
        text
    )

    if match:
        longitude = float(match.group(1))
        latitude = float(match.group(2))
        return pd.Series([longitude, latitude])

    return pd.Series([np.nan, np.nan])

cleaned[["longitude", "latitude"]] = (
    cleaned["point"].apply(extract_coordinates)
)

### Convert rent and square footage ranges

In [ ]:
def get_midpoint(value):
    if pd.isna(value):
        return np.nan

    numbers = re.findall(r"\d[\d,]*", str(value))

    numbers = [
        float(number.replace(",", ""))
        for number in numbers
    ]

    if len(numbers) >= 2:
        return (numbers[0] + numbers[1]) / 2

    return np.nan

cleaned["rent_midpoint"] = (
    cleaned["monthly_rent"].apply(get_midpoint)
)

cleaned["square_footage_midpoint"] = (
    cleaned["square_footage"].apply(get_midpoint)
)

### Remove missing target values

In [ ]:
# Rent is the target variable
eda = cleaned.dropna(
    subset=["rent_midpoint"]
).copy()

In [ ]:
print("\nColumn information:")
eda.info()


Column information:
<class 'pandas.core.frame.DataFrame'>
Index: 477394 entries, 0 to 551550
Data columns (total 32 columns):
 #   Column                               Non-Null Count   Dtype  
---  ------                               --------------   -----  
 0   unique_id                            477394 non-null  int64  
 1   block_num                            477394 non-null  string 
 2   unit_count                           477394 non-null  float64
 3   submission_year                      477394 non-null  int64  
 4   occupancy_type                       477394 non-null  object 
 5   occupancy_or_vacancy_date            441557 non-null  object 
 6   occupancy_or_vacancy_date_year       459007 non-null  object 
 7   bedroom_count                        476614 non-null  object 
 8   bathroom_count                       476955 non-null  object 
 9   square_footage                       477394 non-null  object 
 10  monthly_rent                         477394 non-null  object 
 1

### Missing values

In [ ]:
print("\nMissing values:")
print(
    eda.isnull()
       .sum()
       .sort_values(ascending=False)
)


Missing values:
vacancy_date                           475864
past_occupancy                         116573
occupancy_or_vacancy_date               35837
occupancy_or_vacancy_date_year          18387
year_property_built                     13957
property_age                            13957
square_footage_midpoint                 10523
bedroom_count                             780
bedroom_count_original                    780
bathroom_count                            439
bathroom_count_original                   439
street_name                                57
analysis_neighborhood                      13
supervisor_district                        10
submission_year                             0
unit_count                                  0
base_rent_includes_refuse_recycling         0
base_rent_includes_electricity              0
base_rent_includes_natural_gas              0
base_rent_includes_water_sewer              0
square_footage                              0
monthly_rent     

### Duplicate rows

In [ ]:
print("\nDuplicate rows:")
print(eda.duplicated().sum())

display(eda.head())


Duplicate rows:
0


,unique_id,block_num,unit_count,submission_year,occupancy_type,occupancy_or_vacancy_date,occupancy_or_vacancy_date_year,bedroom_count,bathroom_count,square_footage,monthly_rent,base_rent_includes_water_sewer,base_rent_includes_natural_gas,base_rent_includes_electricity,base_rent_includes_refuse_recycling,past_occupancy,vacancy_date,signature_date,year_property_built,point,analysis_neighborhood,supervisor_district,property_age,bedroom_count_original,bedroom_category,bathroom_count_original,bathroom_category,street_name,longitude,latitude,rent_midpoint,square_footage_midpoint
0,-8888931167805846110,3180,173.0,2023,Occupied by non-owner,2021/06/14,2021,Two-Bedroom,Two bathrooms,1001-1250 Sq.Ft,$3751-$4000,0,0,0,0,No,NaN,2023/02/28,2012.0,POINT (-122.455200494 37.722760282),Oceanview/Merced/Ingleside,11.0,11.0,Two-Bedroom,2 Bedrooms,Two bathrooms,2 Bathrooms,BRIGHTON AVE,-122.455200,37.722760,3875.5,1125.5
1,-8728105716412668324,3180,173.0,2025,Occupied by non-owner,2023/10/14,2023,One-Bedroom,One bathroom,751-1000 Sq.Ft,$3001-$3250,0,0,0,0,Yes,NaN,2025/02/23,2012.0,POINT (-122.455062194 37.722760694),Oceanview/Merced/Ingleside,11.0,13.0,One-Bedroom,1 Bedroom,One bathroom,1 Bathroom,BRIGHTON AVE,-122.455062,37.722761,3125.5,875.5
2,-8664205031566791701,0145,14.0,2023,Occupied by non-owner,1976/05/01,1976,One-Bedroom,One bathroom,501-750 Sq.Ft,$501-$750,1,0,0,1,No,NaN,2023/02/27,1907.0,POINT (-122.406233728 37.798263576),North Beach,3.0,116.0,One-Bedroom,1 Bedroom,One bathroom,1 Bathroom,ROMOLO ST,-122.406234,37.798264,625.5,625.5
3,-8198093964871002724,0285,70.0,2023,Occupied by non-owner,2022/12/15,2022,Studio,One bathroom,251-500 Sq.Ft,$1001-$1250,1,0,0,1,No,NaN,2023/02/24,1911.0,POINT (-122.407883607 37.793610704),Chinatown,3.0,112.0,Studio,Studio,One bathroom,1 Bathroom,STOCKTON ST,-122.407884,37.793611,1125.5,375.5
4,-8126887673081079071,3180,173.0,2022,Occupied by non-owner,2021/07/25,2021,Two-Bedroom,Two bathrooms,1001-1250 Sq.Ft,$4001-$4250,0,0,0,0,Yes,NaN,2022/06/30,2012.0,POINT (-122.455200494 37.722760282),Oceanview/Merced/Ingleside,11.0,10.0,Two-Bedroom,2 Bedrooms,Two bathrooms,2 Bathrooms,BRIGHTON AVE,-122.455200,37.722760,4125.5,1125.5


In [ ]:
drop_columns = [
    # Identifier; keep separately only for tracking
    # "unique_id",

    # Raw columns replaced by cleaned versions
    "monthly_rent",
    "square_footage",
    "bedroom_count",
    "bedroom_count_original",
    "bathroom_count",
    "bathroom_count_original",

    # Geographic object replaced by latitude and longitude
    "point",

    # Mostly missing
    "vacancy_date",

    # Redundant with occupancy_or_vacancy_date
    "occupancy_or_vacancy_date_year",

    # Redundant with property_age
    "year_property_built"
]

eda_final = cleaned.drop(
    columns=drop_columns,
    errors="ignore"
).copy()

print("Final shape:", eda_final.shape)
eda_final.info()

Final shape: (548919, 22)
<class 'pandas.core.frame.DataFrame'>
Index: 548919 entries, 0 to 551550
Data columns (total 22 columns):
 #   Column                               Non-Null Count   Dtype  
---  ------                               --------------   -----  
 0   unique_id                            548919 non-null  int64  
 1   block_num                            548917 non-null  string 
 2   unit_count                           548917 non-null  float64
 3   submission_year                      548919 non-null  int64  
 4   occupancy_type                       548919 non-null  object 
 5   occupancy_or_vacancy_date            448314 non-null  object 
 6   base_rent_includes_water_sewer       548919 non-null  int64  
 7   base_rent_includes_natural_gas       548919 non-null  int64  
 8   base_rent_includes_electricity       548919 non-null  int64  
 9   base_rent_includes_refuse_recycling  548919 non-null  int64  
 10  past_occupancy                       397178 non-null  objec

### Convert date

In [ ]:
date_columns = [
    "occupancy_or_vacancy_date",
    "signature_date"
]

for col in date_columns:
    eda_final[col] = pd.to_datetime(
        eda_final[col],
        errors="coerce"
    )

### Display first five rows with every column

In [ ]:
# Display first five rows with every column
display(eda_final.head())



,unique_id,block_num,unit_count,submission_year,occupancy_type,occupancy_or_vacancy_date,base_rent_includes_water_sewer,base_rent_includes_natural_gas,base_rent_includes_electricity,base_rent_includes_refuse_recycling,past_occupancy,signature_date,analysis_neighborhood,supervisor_district,property_age,bedroom_category,bathroom_category,street_name,longitude,latitude,rent_midpoint,square_footage_midpoint
0,-8888931167805846110,3180,173.0,2023,Occupied by non-owner,2021-06-14,0,0,0,0,No,2023-02-28,Oceanview/Merced/Ingleside,11.0,11.0,2 Bedrooms,2 Bathrooms,BRIGHTON AVE,-122.455200,37.722760,3875.5,1125.5
1,-8728105716412668324,3180,173.0,2025,Occupied by non-owner,2023-10-14,0,0,0,0,Yes,2025-02-23,Oceanview/Merced/Ingleside,11.0,13.0,1 Bedroom,1 Bathroom,BRIGHTON AVE,-122.455062,37.722761,3125.5,875.5
2,-8664205031566791701,0145,14.0,2023,Occupied by non-owner,1976-05-01,1,0,0,1,No,2023-02-27,North Beach,3.0,116.0,1 Bedroom,1 Bathroom,ROMOLO ST,-122.406234,37.798264,625.5,625.5
3,-8198093964871002724,0285,70.0,2023,Occupied by non-owner,2022-12-15,1,0,0,1,No,2023-02-24,Chinatown,3.0,112.0,Studio,1 Bathroom,STOCKTON ST,-122.407884,37.793611,1125.5,375.5
4,-8126887673081079071,3180,173.0,2022,Occupied by non-owner,2021-07-25,0,0,0,0,Yes,2022-06-30,Oceanview/Merced/Ingleside,11.0,10.0,2 Bedrooms,2 Bathrooms,BRIGHTON AVE,-122.455200,37.722760,4125.5,1125.5


### Display dataset information

In [ ]:
# Display dataset information
eda_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 548919 entries, 0 to 551550
Data columns (total 22 columns):
 #   Column                               Non-Null Count   Dtype         
---  ------                               --------------   -----         
 0   unique_id                            548919 non-null  int64         
 1   block_num                            548917 non-null  string        
 2   unit_count                           548917 non-null  float64       
 3   submission_year                      548919 non-null  int64         
 4   occupancy_type                       548919 non-null  object        
 5   occupancy_or_vacancy_date            448158 non-null  datetime64[ns]
 6   base_rent_includes_water_sewer       548919 non-null  int64         
 7   base_rent_includes_natural_gas       548919 non-null  int64         
 8   base_rent_includes_electricity       548919 non-null  int64         
 9   base_rent_includes_refuse_recycling  548919 non-null  int64         
 10  p